# Brick 1 Continual Learning — Colab Pro+ Run

Warm-start LoRA from Brick 0 v5 on Qwen2.5-0.5B, mix 60% replay + 40% boost, run all 6 gates.

**Inputs (pulled from HF):**
- Adapter: `issdandavis/tongue-table-lora-brick0-v5`
- Data: `issdandavis/scbe-drill-langues-full`

**Outputs (pushed to HF):**
- Brick 1 adapter + reports → `issdandavis/tongue-table-lora-brick1-v1`

Build order per `docs/BRICK1_CONTINUAL_LEARNING_PLAN.md` §5:
1. `scripts/build_brick1_boost.py` → boost file + manifest
2. `scripts/train_brick1_continual.py` → LoRA fit with resume + cosine LR
3. `scripts/eval_brick1_gates.py` → gate report (exit 0/1/2/3)

## 1. Environment setup

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/issdandavis/SCBE-AETHERMOORE.git'
BRANCH = 'neurogolf/ant-colony-solvers'
REPO_DIR = '/content/SCBE-AETHERMOORE'

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'])

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())
subprocess.check_call(['git', 'log', '-1', '--oneline'])

In [ ]:
!pip install -q 'transformers>=4.46' 'peft>=0.13' 'accelerate>=1.0' 'datasets>=3.0' 'bitsandbytes>=0.44' 'huggingface_hub>=0.26' 'safetensors' 'trl>=0.11'

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2. HF auth (paste token with `write` scope — used for pushing Brick 1 artifacts back)

In [ ]:
from huggingface_hub import login, whoami
from getpass import getpass

if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass('HF token (needs write scope): ')
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('auth:', whoami()['name'])

## 3. Pull Brick 0 adapter + drill data from HF

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
from pathlib import Path

brick0_local = Path('artifacts/tongue-table-lora-brick0-v5/lora_final')
brick0_local.parent.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id='issdandavis/tongue-table-lora-brick0-v5',
    repo_type='model',
    local_dir=str(brick0_local),
)
print('brick0 adapter at:', brick0_local)
print(sorted(p.name for p in brick0_local.iterdir()))

drill_local = Path('data/tongue_drill/drill_langues_full.jsonl')
drill_local.parent.mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id='issdandavis/scbe-drill-langues-full',
    repo_type='dataset',
    filename='drill_langues_full.jsonl',
    local_dir=str(drill_local.parent),
)
print('drill data rows:', sum(1 for _ in open(drill_local, encoding='utf-8')))

## 4. Also fetch the Brick 0 eval reports (baseline for G6 tolerance gate)

In [ ]:
# These are in the repo under artifacts/... but artifacts/ is gitignored.
# Plan B: regenerate them here from the Brick 0 adapter + drill data.
# The gate script needs:
#   artifacts/tongue-table-lora-brick0-v5/final_structural_benchmark.json
#   artifacts/tongue-table-lora-brick0-v5/final_drill_map_eval.json

import json
from pathlib import Path

needed = [
    Path('artifacts/tongue-table-lora-brick0-v5/final_structural_benchmark.json'),
    Path('artifacts/tongue-table-lora-brick0-v5/final_drill_map_eval.json'),
]
missing = [p for p in needed if not p.exists()]
print('missing baseline reports:', missing)

if missing:
    print('regenerating baseline reports from Brick 0 adapter...')
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from scripts.train.lora_tongue_table import run_drill_map_eval, run_structural_benchmark_eval

    base_model = 'Qwen/Qwen2.5-0.5B'
    tok = AutoTokenizer.from_pretrained(base_model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(base_model, low_cpu_mem_usage=True, dtype=torch.float16)
    mdl = PeftModel.from_pretrained(base, 'artifacts/tongue-table-lora-brick0-v5/lora_final', is_trainable=False).to('cuda')
    mdl.eval()

    struct = run_structural_benchmark_eval(
        mdl, tok,
        data_path='data/tongue_drill/drill_langues_full.jsonl',
        split='holdout', holdout_mod=10, holdout_bucket=0, max_per_stage=12,
    )
    needed[0].write_text(json.dumps(struct, indent=2), encoding='utf-8')

    drill = run_drill_map_eval(
        mdl, tok,
        data_path='data/tongue_drill/drill_langues_full.jsonl',
        split='holdout', holdout_mod=10, holdout_bucket=0, max_per_cell=10, max_length=256,
    )
    needed[1].write_text(json.dumps(drill, indent=2), encoding='utf-8')
    print('baseline reports regenerated')

print('structural summary:', json.loads(needed[0].read_text())['summary'])
print('drill summary:', json.loads(needed[1].read_text())['_summary'])

## 5. Step 1 — Build the Brick 1 boost file

In [ ]:
!python scripts/build_brick1_boost.py

## 6. Step 2 — Continual LoRA fit (60/40 mix, cosine LR, early stop)

In [ ]:
!python scripts/train_brick1_continual.py

## 7. Step 3 — Gate eval (exit 0 = green, 1 = amber, 2 = red, 3 = setup error)

In [ ]:
import subprocess
proc = subprocess.run(['python', 'scripts/eval_brick1_gates.py'])
print('gate exit code:', proc.returncode)
print({0: 'GREEN — all gates pass', 1: 'AMBER — below target, above floor', 2: 'RED — below floor', 3: 'SETUP ERROR'}.get(proc.returncode, 'UNKNOWN'))

In [ ]:
import json
report = json.loads(open('artifacts/tongue-table-lora-brick1-v1/brick1_gate_report.json').read())
print('overall_status:', report['overall_status'])
for g in report['gates']:
    v = g.get('brick1_value')
    vs = f'{v:.4f}' if isinstance(v, float) else 'N/A'
    print(f"  [{g['status']:7s}] {g['gate']:22s} brick1={vs}  {g['reason']}")

## 8. Push Brick 1 artifacts back to HF

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
target = 'issdandavis/tongue-table-lora-brick1-v1'
api.create_repo(repo_id=target, repo_type='model', exist_ok=True, private=False)
api.upload_folder(
    folder_path='artifacts/tongue-table-lora-brick1-v1',
    repo_id=target,
    repo_type='model',
    commit_message='Brick 1 continual LoRA + gate report',
)
print('uploaded:', f'https://huggingface.co/{target}')